# Evaluating Metadata Quality

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from prettytable import PrettyTable

In [2]:
working_dir = Path("../papers/face-eval")
evaluation_dir = working_dir / "fg" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)
automatic_annotations_path = working_dir / "fg/output/dataset_table.csv"
manual_annotations_path = "../manual_annotations/face/dataset_table.csv"

In [3]:
manual_df = pd.read_csv(manual_annotations_path)
automatic_df = pd.read_csv(automatic_annotations_path)

In [4]:
def eval_correct(auto, manual):
    return (auto == manual) \
           | (auto.isna() & manual.isna())

In [5]:
def eval_metrics(auto, manual, correct):
    n1 = (auto.notna() & manual.notna() & correct).sum()  # right answer
    n2 = (auto.notna() & manual.notna() & ~correct).sum()  # wrong answer
    n3 = (auto.isna() & manual.notna()).sum()  # missed
    n4 = (auto.notna() & manual.isna()).sum()  # hallucinated
    n5 = (auto.isna() & manual.isna()).sum()  # correctly abstained
    return n1, n2, n3, n4, n5

In [6]:
def print_abstension_confusion_matrix(counts):
    n1, n2, n3, n4, n5 = counts
    t = PrettyTable()
    t.field_names = ["True \\ Pred"] + ["Correctly\nanswered", "Incorrectly\nanswered", "Abstained"]
    t.add_row(["No abstention", n1, n2, n3])
    t.add_row(["Abstention", None, n4, n5])
    print(t)

In [ ]:
def print_metrics(counts):
    n1, n2, n3, n4, n5 = counts
    r_acc = n1 / (n1 + n2 + n4) if (n1 + n2 + n4) else 0
    urup = 1- (n5/(n2+n4+n5))
    acc = (n1 + n5) / (n1 + n2 + n3 + n4 + n5)
    coverage = (n1 + n2) / (n1 + n2 + n3)
    # rec = tp / (tp + wrong + fn) if (tp + wrong + fn) else 0
    # accuracy = (tp + tn) / (tp + wrong + fp + fn + tn)
    print(f"Reliable Accuracy: {r_acc:.2f}")
    print(f"BAR: {coverage:.2f}")
    print(f"ACC: {acc:.2f}")

In [10]:
auto_page = automatic_df["project_page"]
manual_page = manual_df["project_page"]
correct = eval_correct(automatic_df["project_page"], manual_df["project_page"])
counts = eval_metrics(automatic_df["project_page"], manual_df["project_page"], correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     56    |      2      |     3     |
|   Abstention  |    None   |      0      |     5     |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.97

BAR: 0.95
ACC: 0.92


In [8]:
automatic_license = automatic_df["data_license"] \
                    .combine_first(automatic_df["website_data_license"])
                    # .combine_first(automatic_df["website_usage_agreement"])
manual_license = manual_df["data_license"] \
                 .combine_first(manual_df["website_data_license"])
                #  .combine_first(automatic_df["website_usage_agreement"])
license_correct = eval_correct(automatic_license, manual_license)
counts = eval_metrics(automatic_license, manual_license, license_correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     17    |      1      |     15    |
|   Abstention  |    None   |      0      |     33    |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.94

BAR: 0.55
ACC: 0.76


In [9]:
automatic_agreement = automatic_df["data_license"] \
                    .combine_first(automatic_df["website_data_license"]) \
                    .combine_first(automatic_df["website_usage_agreement"])
manual_agreement = manual_df["data_license"] \
                 .combine_first(manual_df["website_data_license"]) \
                 .combine_first(manual_df["website_usage_agreement"])
# agreement_df = automatic_agreement.to_frame() \
#                   .join(manual_agreement, lsuffix="_auto", rsuffix="_manual")
correct = pd.read_csv(evaluation_dir / "agreement_manual.csv")["correct"]
counts = eval_metrics(automatic_agreement, manual_agreement, correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     26    |      2      |     15    |
|   Abstention  |    None   |      0      |     23    |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.93

BAR: 0.65
ACC: 0.74
